# Rede de Transferências de Futebol (Transfermarkt)
### Trabalho Prático de Grafos usando a biblioteca `pgraph`

**Problema:** modelar o mercado de transferências de futebol como um grafo direcionado e ponderado, onde cada vértice é um clube e cada aresta A->B indica que pelo menos um jogador foi transferido do clube A para o clube B.

**Perguntas que queremos responder:**
1. Quais clubes concentram maior *prestígio* no mercado (recebem jogadores de clubes já valorizados)? -> **PageRank**
2. Quais clubes funcionam como *pontes estruturais* entre mercados/ligas diferentes (ex: comprar de um continente/liga menor e revender para ligas grandes)? -> **Centralidade de Intermediação (Betweenness)**

**Fonte de dados:** dataset público "Football Data from Transfermarkt" (Kaggle), arquivos `clubs.csv` e `transfers.csv`.

---
> **Links para os datasets** 
> - clubs.csv: https://www.kaggle.com/datasets/davidcariboo/player-scores?select=clubs.csv
> - transfers.csv: https://www.kaggle.com/datasets/davidcariboo/player-scores?select=transfers.csv


---
> **Como usar este notebook no Google Colab:**
> 1. Faça upload de `clubs.csv` e `transfers.csv`.
> 2. Rode as células em ordem.
> 3. A biblioteca `pgraph` é instalada direto do seu repositório GitHub.


## 1. Setup  instalar a biblioteca `pgraph` e dependências

In [1]:
!pip install -q git+https://github.com/Pedro-io/pgraph.git
!pip install -q pandas


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from collections import deque

from graph import AdjacencyListGraph

# a lib usa loguru para debug interno. Estou removendo para uma saída mais limpa
from loguru import logger as _pgraph_logger
_pgraph_logger.remove()


## 2. Carregar os dados

In [4]:
# Atualize os caminhos dos arquivos CSV conforme necessário
clubs = pd.read_csv("/content/data/clubs.csv")
transfers = pd.read_csv("/content/data/transfers.csv")

print("clubs:", clubs.shape)
print("transfers:", transfers.shape)
clubs.head()


clubs: (796, 17)
transfers: (175043, 10)


,club_id,club_code,name,domestic_competition_id,total_market_value,squad_size,average_age,foreigners_number,foreigners_percentage,national_team_players,stadium_name,stadium_seats,net_transfer_record,coach_name,last_season,filename,url
0,10,arminia-bielefeld,Arminia Bielefeld,L1,NaN,27,25.3,15,55.6,4,SchücoArena,26515,+€5.90m,NaN,2021,../data/raw/transfermarkt-scraper/2021/clubs.j...,https://www.transfermarkt.co.uk/arminia-bielef...
1,10004,paris-fc,Paris Football Club,FR1,NaN,31,28.8,17,54.8,9,Stade Jean Bouin,19904,€-72.30m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/paris-fc/start...
2,10010,esporte-clube-bahia,Esporte Clube Bahia,BRA1,NaN,31,26.8,6,19.4,3,Arena Fonte Nova,47364,+€8.14m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/esporte-clube-...
3,1003,leicester-city,Leicester City,GB1,NaN,29,25.9,17,58.6,9,King Power Stadium,32259,+€57.30m,Steve Cooper,2024,../data/raw/transfermarkt-scraper/2024/clubs.j...,https://www.transfermarkt.co.uk/leicester-city...
4,1005,us-lecce,Unione Sportiva Lecce,IT1,NaN,27,25.3,23,85.2,13,Ettore Giardiniero,31559,+€8.62m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/us-lecce/start...


## 3. Limpeza e filtragem dos dados

Decisões tomadas:

- **Apenas clubes presentes em `clubs.csv`** (796 clubes de competições cobertas pelo dataset)  o `transfers.csv` sozinho referencia ~17,5 mil "clubes" (muitos são times menores/desconhecidos sem outros dados associados). Restringir aos 796 clubes conhecidos mantém o grafo grande o suficiente pra análise.
- **Remoção de self-loops** (`from_club_id == to_club_id`).
- **Corte de datas futuras/inconsistentes** (`transfer_date <= hoje`)  o dataset bruto contém algumas poucas datas futuras (provavelmente inconsistências de scraping ou anotação).
- **Agregação de múltiplas transferências entre o mesmo par de clubes** em uma única aresta ponderada  a lib exige `add_edge` idempotente (grafo simples), e semanticamente queremos a *intensidade* da relação, não uma lista de eventos.


In [5]:
known_ids = set(clubs["club_id"])

df = transfers[
    transfers["from_club_id"].isin(known_ids) &
    transfers["to_club_id"].isin(known_ids)
].copy()

df = df[df["from_club_id"] != df["to_club_id"]]

df["transfer_date"] = pd.to_datetime(df["transfer_date"], errors="coerce")
HOJE = pd.Timestamp.today().normalize()
df = df[df["transfer_date"] <= HOJE]

print("transferências após filtragem:", len(df))
print("clubes distintos envolvidos:", pd.concat([df["from_club_id"], df["to_club_id"]]).nunique())


transferências após filtragem: 47654
clubes distintos envolvidos: 796


## 4. Agregação em arestas ponderadas

Para cada par (clube_origem, clube_destino), agregamos:
- `n_transfers`: quantidade de transferências entre esse par (peso principal da aresta  mais robusto que valor de mercado, já que ~60% das taxas no dataset são nulas/zero: empréstimos, fim de contrato, valores não divulgados)
- `total_fee`: soma das taxas conhecidas


In [6]:
edges = (
    df.groupby(["from_club_id", "to_club_id"])
      .agg(n_transfers=("player_id", "count"),
           total_fee=("transfer_fee", lambda s: s.fillna(0).sum()))
      .reset_index()
)

print("arestas (pares únicos de clubes):", len(edges))
edges.sort_values("n_transfers", ascending=False).head(10)


arestas (pares únicos de clubes): 29113


,from_club_id,to_club_id,n_transfers,total_fee
26167,11194,419,45,14600000.0
8920,419,11194,34,0.0
12187,697,62,27,5050000.0
2218,62,697,25,710000.0
14671,1010,410,25,56800000.0
7143,338,10690,25,1070000.0
8468,410,1010,25,21900000.0
26012,10690,338,22,6200000.0
24502,6993,660,22,0.0
11684,660,6993,20,0.0


## 5. Mapear clube -> índice inteiro

In [7]:
clubs_in_graph = sorted(set(edges["from_club_id"]) | set(edges["to_club_id"]))

id_to_index = {club_id: i for i, club_id in enumerate(clubs_in_graph)}
index_to_name = {
    i: clubs.loc[clubs["club_id"] == club_id, "name"].values[0]
    for club_id, i in id_to_index.items()
}

N = len(clubs_in_graph)
print("total de vértices no grafo:", N)


total de vértices no grafo: 796


## 6. Construir o grafo usando a biblioteca `pgraph`

O grafo "oficial" do trabalho é este objeto `g`, construído inteiramente com a API criada na biblioteca. É nele que validamos propriedades estruturais (`is_connected`, graus, etc.) e é dele que exportamos para o Gephi no final.


In [15]:
g = AdjacencyListGraph(N)

for row in edges.itertuples(index=False):
    u = id_to_index[row.from_club_id]
    v = id_to_index[row.to_club_id]
    g.add_edge(u, v)
    g.set_edge_weight(u, v, float(row.n_transfers))

print("vértices:", g.get_vertex_count())
print("arestas:", g.get_edge_count())
print("grafo conectado?", g.is_connected())


vértices: 796
arestas: 29113
grafo conectado? False


## 7. Estrutura auxiliar de adjacência

O `pgraph` expõe apenas consultas pontuais (`has_edge`, `get_edge_weight`, graus), sem um método para iterar os vizinhos de um vértice. Para os algoritmos abaixo (PageRank e Betweenness), que precisam varrer vizinhanças repetidamente, construímos uma lista de adjacência auxiliar **a partir dos mesmos dados já validados no grafo `g`**  ou seja, é apenas uma estrutura de indexação para performance, o grafo canônico do trabalho continua sendo `g`.


In [16]:
out_adj = {i: [] for i in range(N)}
for _, row in edges.iterrows():
    u = id_to_index[row["from_club_id"]]
    v = id_to_index[row["to_club_id"]]
    w = g.get_edge_weight(u, v)  # lê o peso direto do grafo oficial, garantindo consistência
    out_adj[u].append((v, w))

print("adjacência construída para", len(out_adj), "vértices")


adjacência construída para 796 vértices


## 8. Algoritmo 1  PageRank (direcionado, ponderado)

PageRank (Brin & Page, 1998) mede importância propagada: um clube é "importante" se recebe jogadores de clubes que já são importantes  não apenas se recebe muitos jogadores. Aqui, o peso da aresta (número de transferências) funciona como a probabilidade de transição: quanto mais transferências histórias entre A e B, maior a fração do "prestígio" de A que flui para B.

Fórmula (com tratamento de nós sem saída  *dangling nodes*):

PR(v) = (1-d)/N + d · [ Σ_u PR(u)·w(u,v)/Σw(u,·)  +  (massa dos dangling nodes)/N ]


In [17]:
def pagerank(out_adj, N, damping=0.85, max_iter=100, tol=1e-8):
    PR = [1.0 / N] * N
    for iteration in range(max_iter):
        new_PR = [(1 - damping) / N] * N

        # massa de PageRank de vértices sem arestas de saída (dangling nodes),
        # redistribuída uniformemente para não "vazar" do sistema
        dangling_sum = sum(PR[u] for u in range(N) if not out_adj.get(u))

        for u in range(N):
            neighbors = out_adj.get(u, [])
            total_w = sum(w for _, w in neighbors)
            if total_w == 0:
                continue
            pu = PR[u]
            for v, w in neighbors:
                new_PR[v] += damping * pu * (w / total_w)

        add = damping * dangling_sum / N
        for v in range(N):
            new_PR[v] += add

        diff = sum(abs(new_PR[i] - PR[i]) for i in range(N))
        PR = new_PR
        if diff < tol:
            break

    return PR, iteration + 1


PR, n_iter = pagerank(out_adj, N)
print(f"PageRank convergiu em {n_iter} iterações (soma = {sum(PR):.4f}, deve ser ≈ 1.0)")


PageRank convergiu em 43 iterações (soma = 1.0000, deve ser ≈ 1.0)


In [18]:
top_pagerank = sorted(range(N), key=lambda i: -PR[i])[:15]

print(f"{'Clube':40s} PageRank")
print("-" * 55)
for i in top_pagerank:
    print(f"{index_to_name[i]:40s} {PR[i]:.5f}")


Clube                                    PageRank
-------------------------------------------------------
Olympiakos Syndesmos Filathlon Peiraios  0.00518
Chelsea Football Club                    0.00435
Football Club Internazionale Milano S.p.A. 0.00432
FC Shakhtar Donetsk                      0.00422
Genoa Cricket and Football Club          0.00416
Sport Lisboa e Benfica                   0.00415
Associazione Sportiva Roma               0.00405
Fenerbahçe Spor Kulübü                   0.00386
Futebol Clube do Porto                   0.00381
Udinese Calcio                           0.00367
Associazione Calcio Fiorentina           0.00361
Juventus Football Club                   0.00361
Futbolniy Klub Dynamo Kyiv               0.00360
Manchester City Football Club            0.00355
Galatasaray Spor Kulübü                  0.00352


## 9. Algoritmo 2  Centralidade de Intermediação (Betweenness, Brandes)

Betweenness mede em quantos caminhos mais curtos entre outros pares de clubes um clube aparece "no meio"  ou seja, quão frequentemente ele funciona como ponte/corredor obrigatório de fluxo entre outras partes do grafo. Aqui usamos a versão **não ponderada** (estrutural, baseada em BFS): o que importa é a topologia de conexões (existe ou não rota de transferência), não a intensidade de cada uma.
Implementação: algoritmo de Brandes (2001), adaptado para grafos direcionados (sem dividir por 2 no final, como se faz no caso não-direcionado).


In [19]:
def betweenness_centrality(out_adj, N):
    CB = [0.0] * N

    for s in range(N):
        dist = [-1] * N
        sigma = [0] * N
        dist[s] = 0
        sigma[s] = 1
        Q = deque([s])
        S = []                      # ordem de visitação (para acumulação reversa)
        preds = [[] for _ in range(N)]

        while Q:
            v = Q.popleft()
            S.append(v)
            for w, _ in out_adj.get(v, []):
                if dist[w] < 0:
                    dist[w] = dist[v] + 1
                    Q.append(w)
                if dist[w] == dist[v] + 1:
                    sigma[w] += sigma[v]
                    preds[w].append(v)

        delta = [0.0] * N
        while S:
            w = S.pop()
            for v in preds[w]:
                if sigma[w] != 0:
                    delta[v] += (sigma[v] / sigma[w]) * (1 + delta[w])
            if w != s:
                CB[w] += delta[w]

    return CB


CB = betweenness_centrality(out_adj, N)
print("Betweenness calculado para todos os", N, "vértices.")


Betweenness calculado para todos os 796 vértices.


In [20]:
top_betweenness = sorted(range(N), key=lambda i: -CB[i])[:15]

print(f"{'Clube':40s} Betweenness")
print("-" * 60)
for i in top_betweenness:
    print(f"{index_to_name[i]:40s} {CB[i]:.1f}")


Clube                                    Betweenness
------------------------------------------------------------
FC Shakhtar Donetsk                      12631.4
Futbolniy Klub Dynamo Kyiv               11983.5
Olympiakos Syndesmos Filathlon Peiraios  11222.1
Sport Lisboa e Benfica                   10449.7
The Celtic Football Club                 7185.7
Fodbold Club Midtjylland                 6733.2
Futebol Clube do Porto                   6731.5
Sporting Clube de Portugal               6562.3
Udinese Calcio                           6541.3
Beşiktaş Jimnastik Kulübü                6531.9
Panthessalonikios Athlitikos Omilos Konstantinoupoliton 5999.7
Fudbalski klub Crvena zvezda Beograd     5987.5
Genoa Cricket and Football Club          5905.8
Fenerbahçe Spor Kulübü                   5808.6
Royal Standard Club de Liège             5679.3


## 11. Salvar métricas no grafo oficial e exportar para o Gephi

In [21]:
# guarda as métricas calculadas como peso do vértice na estrutura oficial da lib
# (aqui usamos set_vertex_weight para o PageRank; se a lib permitir múltiplos atributos,
# ajuste conforme docs/api.md  caso só exista um peso por vértice, exporte um CSV auxiliar
# com as duas métricas para importar como atributos no Gephi)

for i in range(N):
    g.set_vertex_weight(i, PR[i])

g.export_to_gephi("transfer_network.gexf")
print("Exportado: transfer_network.gexf")

# exporta também uma tabela de atributos (PageRank + Betweenness + nome) para importar
# manualmente no Gephi via "Data Laboratory > Import Spreadsheet", caso a lib exporte
# só um atributo de peso por vértice
attrs = pd.DataFrame({
    "id": list(range(N)),
    "label": [index_to_name[i] for i in range(N)],
    "pagerank": PR,
    "betweenness": CB,
})
attrs.to_csv("club_metrics.csv", index=False)
print("Exportado: club_metrics.csv (para importar como atributos no Gephi, se necessário)")


Exportado: transfer_network.gexf
Exportado: club_metrics.csv (para importar como atributos no Gephi, se necessário)
